In [1]:
import os
import gc
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from transformers import (
    GenerationConfig,
    MBartForConditionalGeneration,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer
)
import re
import torch
import pandas as pd
import unicodedata
from pathlib import Path
from sklearn.model_selection import train_test_split
from datasets import Dataset

In [2]:
TRAIN_SOURCE_PATH = Path('data/transcription_train.txt')
TRAIN_TARGET_PATH = Path('data/english_train.txt')
TEST_SOURCE_PATH = Path('data/transcription_validation.txt')
TEST_TARGET_PATH = Path('data/english_validation.txt')
CLEAN_TRAIN_OUTPUT = Path('data/train_cleaned.csv')
CLEAN_VAL_OUTPUT = Path('data/validation_cleaned.csv')
CLEAN_TEST_OUTPUT = Path('data/test_cleaned.csv')
TRAIN_SPLIT_RATIO = 0.9
RANDOM_STATE = 42
BLOCK_MIN_LINES = 2
BLOCK_MAX_LINES = 3
MAX_LENGTH_RATIO = 3.0
MIN_ID_SIGN_COUNT = 3
USE_BLOCKS = True
FAST_TEST_RUN = False
BLOCK_LENGTH = 192 # 128, 160, 192, 256 and 384 were tested
MAX_TRAIN_SAMPLES = 1200 if FAST_TEST_RUN else None
MAX_VAL_SAMPLES = 200 if FAST_TEST_RUN else None
MAX_TEST_SAMPLES = 200 if FAST_TEST_RUN else None
MAX_SOURCE_LENGTH = BLOCK_LENGTH
MAX_TARGET_LENGTH = BLOCK_LENGTH
NUM_TRAIN_EPOCHS = 1 if FAST_TEST_RUN else 15
MAX_TRAIN_STEPS = 200 if FAST_TEST_RUN else -1
TRAINING_OUTPUT_DIR = f'./mbart-akkadian-to-en-test-no-blocks-len-{BLOCK_LENGTH}-full-2' if not USE_BLOCKS else f'./mbart-akkadian-to-en-test-len-{BLOCK_LENGTH}-full-2'

In [3]:
# translation table for mapping normal digits to Unicode subscript digits
# translation tables define how to replace characters in a string using the str.translate() method 
SUBSCRIPT_DIGITS = str.maketrans('0123456789', '₀₁₂₃₄₅₆₇₈₉')

# Normalize determinatives between curly braces with expanded scholarly variations
def _normalize_determinative(match):
    content = match.group(1).strip().lower()
    
    # Remove internal @v or ~v flags (e.g., {lu2@v} → {lu2})
    content = re.sub(r'[@~]v', '', content)
    
    # Standardize male markers
    if content in ('1', 'm', 'disz'):
        content = 'm'
    # Standardize female markers
    elif content in ('mi2', 'f', 'munus'):
        content = 'f'
    # Standardize city markers
    elif content in ('iri', 'uru'):
        content = 'uru'
    
    return '{' + content + '}'

# Standardize all subscripts (u2, u_2, u₂) to a single Unicode subscript format (u₂)
def _normalize_subscripts(text):
    def replace(match):
        base = match.group(1)
        index = match.group(2).replace('_', '').translate(SUBSCRIPT_DIGITS)
        return base + index
    # Regex match letters followed by a number with optional underscore
    return re.sub(r'(?<!\w)([A-Za-zšṣṭḫĝŋŠṢṬḪ]+)_?([0-9]+)', replace, text)

# Normalize representations of gaps (..., x, [ ], [..]) in Akkadian texts 
# to standardized tokens <gap> and <big_gap>
def _normalize_akkadian_gaps(text):
    text = re.sub(r'\.{4,}', ' <big_gap> ', text)
    text = re.sub(r'\.{3}', ' <gap> ', text)
    text = re.sub(r'\[(?:\s*\.){0,3}\s*\]', ' <gap> ', text)
    text = re.sub(r'\b(?:x|X)\b', ' <gap> ', text)
    text = re.sub(r'(?<!\w)\[\s*\]\b', ' <gap> ', text)
    text = re.sub(r'(?<!\w)\[\.{1,3}\](?!\w)', ' <gap> ', text)
    text = re.sub(r'(?<!\w)\[\.\.\](?!\w)', ' <gap> ', text)
    return text

# Removes annotation artifacts such as line numbering, obverse/reverse tags, ...
def _strip_scholarly_noise(text):
    text = re.sub(r'(?m)^\s*\d+\.\s*', '', text)  # Leading list numbers (e.g., "1. ")
    text = re.sub(r'(?m)^\s*[ivxlcdm]+\s+\d+\'?\s*', '', text, flags=re.IGNORECASE)  # Leading Roman numeral + number
    text = re.sub(r'(?m)^\s*\d+\s+\d+\'?\s*', '', text)  # Leading double number sequences (line markings)
    text = re.sub(r'\{([^{}]+?)(?:[@~](?:v|obverse|reverse|obv|rev))\}', lambda m: '{' + m.group(1) + '}', text, flags=re.IGNORECASE)  # Strip side tags inside {}
    text = re.sub(r'([^\s{}]+?)(?:[@~](?:v|obverse|reverse|obv|rev))\b', r'\1', text, flags=re.IGNORECASE)  # Strip side tags from words
    text = re.sub(r'(?i)\s*@(?:obverse|reverse|obv|rev)\b', ' ', text)  # Delete standalone side tags
    text = re.sub(r'(?<!\w)[#!?]+(?!\w)', '', text)  # Delete isolated punctuation
    text = re.sub(r'(?i)\bo\s*$', '', text)  # Delete trailing "o" at end of string
    return text

# Normalize Akkadian text by standardizing subscripts, determinatives, gaps, and removing annotation artifacts
def normalize_akkadian(text):
    if pd.isna(text):
        return ''
    text = unicodedata.normalize('NFKC', str(text))
    text = _normalize_subscripts(text)
    text = unicodedata.normalize('NFC', text)
    text = re.sub(r'\{([^}]+)\}', _normalize_determinative, text)
    text = _strip_scholarly_noise(text)
    text = _normalize_akkadian_gaps(text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# Normalize English text by removing parenthetical content, standardizing gaps, and collapsing whitespace
# Lighter clean up since source transliteration is noisier than target translation
def normalize_english(text):
    if pd.isna(text):
        return ''
    text = unicodedata.normalize('NFKC', str(text))
    text = re.sub(r'\([^)]*\)', ' ', text)
    text = re.sub(r'\.{4,}', ' <big_gap> ', text)
    text = re.sub(r'\.{3}', ' <gap> ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# Count identifiable signs in the Akkadian text, ignoring gaps and non-sign tokens
# Estimation of readable source content after normalization
def count_identifiable_signs(text):
    if not text:
        return 0
    tokens = re.findall(r'<big_gap>|<gap>|\{[^}]+\}|[^\s]+', text)
    sign_count = 0
    for token in tokens:
        if token in {'<gap>', '<big_gap>'}:
            continue
        if token.startswith('{') and token.endswith('}'):
            sign_count += 1
            continue
        parts = [part for part in re.split(r'[-\s]+', token) if part and part not in {'#', '!', '?', 'o'}]
        sign_count += len(parts)
    return sign_count

# Filter out pairs where either source or target is empty, 
# source has too few identifiable signs, 
# or length ratio is too extreme
# This avoids training on pairs that are empty, too damaged, or strongly length-misaligned.
def is_meaningful_pair(source_text, target_text):
    if not source_text or not target_text:
        return False
    source_core = source_text.replace('<gap>', '').replace('<big_gap>', '').strip()
    if not source_core:
        return False
    if count_identifiable_signs(source_text) < MIN_ID_SIGN_COUNT:
        return False
    source_length = len(source_text.split())
    target_length = len(target_text.split())
    if source_length == 0 or target_length == 0:
        return False
    length_ratio = max(source_length / target_length, target_length / source_length)
    return length_ratio <= MAX_LENGTH_RATIO

# Determine block sizes for grouping lines based on total number of rows, ensuring blocks of 2-3 lines
def build_block_sizes(total_rows):
    if total_rows < BLOCK_MIN_LINES:
        return []
    if total_rows == 2:
        return [2]
    remainder = total_rows % BLOCK_MAX_LINES
    if remainder == 0:
        return [3] * (total_rows // 3)
    if remainder == 1:
        if total_rows < 4:
            return []
        return [3] * (total_rows // 3 - 1) + [2, 2]
    return [3] * (total_rows // 3) + [2]

# Sentence Reconstruction: Group consecutive lines into blocks of 2-3 lines to create more meaningful training pairs
# mBart benefits from training on longer sequences, and many sentences in the dataset are split across multiple lines.
# Akkadian often puts the verb at the end of the sentence, by grouping lines the model can better understand the relationships
def group_into_blocks(frame):
    if frame.empty:
        return pd.DataFrame(columns=['source_text', 'target_text'])
    block_sizes = build_block_sizes(len(frame))
    grouped_rows = []
    start_index = 0
    for block_size in block_sizes:
        block = frame.iloc[start_index:start_index + block_size]
        source_text = ' '.join(block['source_text'].tolist()).strip()
        target_text = ' '.join(block['target_text'].tolist()).strip()
        if is_meaningful_pair(source_text, target_text):
            grouped_rows.append({'source_text': source_text, 'target_text': target_text})
        start_index += block_size
    return pd.DataFrame(grouped_rows, columns=['source_text', 'target_text'])

def load_clean_parallel_dataset(source_path, target_path, save_path=None, use_blocks=True):
    cleaned_rows = []
    with open(source_path, encoding='utf-8') as source_file, open(target_path, encoding='utf-8') as target_file:
        for source_line, target_line in zip(source_file, target_file):
            source_text = normalize_akkadian(source_line.rstrip('\n'))
            target_text = normalize_english(target_line.rstrip('\n'))
            if is_meaningful_pair(source_text, target_text):
                cleaned_rows.append({'source_text': source_text, 'target_text': target_text})
    cleaned_frame = pd.DataFrame(cleaned_rows, columns=['source_text', 'target_text'])
    if use_blocks:
        final_frame = group_into_blocks(cleaned_frame)
    else:
        final_frame = cleaned_frame
    if save_path is not None:
        final_frame.to_csv(save_path, index=False)
    return final_frame

In [4]:
df_train_cleaned = load_clean_parallel_dataset(
    TRAIN_SOURCE_PATH,
    TRAIN_TARGET_PATH,
    use_blocks=USE_BLOCKS
)

df_train_cleaned = df_train_cleaned.rename(columns={
    'source_text': 'akkadian',
    'target_text': 'english'
})

df_train, df_val = train_test_split(
    df_train_cleaned,
    test_size=1 - TRAIN_SPLIT_RATIO,
    random_state=RANDOM_STATE,
    shuffle=True
)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)

if MAX_TRAIN_SAMPLES is not None and len(df_train) > MAX_TRAIN_SAMPLES:
    df_train = df_train.sample(n=MAX_TRAIN_SAMPLES, random_state=RANDOM_STATE).reset_index(drop=True)

if MAX_VAL_SAMPLES is not None and len(df_val) > MAX_VAL_SAMPLES:
    df_val = df_val.sample(n=MAX_VAL_SAMPLES, random_state=RANDOM_STATE).reset_index(drop=True)

df_train.to_csv(CLEAN_TRAIN_OUTPUT, index=False)
df_val.to_csv(CLEAN_VAL_OUTPUT, index=False)

print('num samples train:', len(df_train))
print('num samples validation:', len(df_val))
print('saved cleaned train dataset to:', CLEAN_TRAIN_OUTPUT)
print('saved cleaned validation dataset to:', CLEAN_VAL_OUTPUT)
print(f'block grouping enabled: {USE_BLOCKS}')
print(f'current block length: {BLOCK_LENGTH}')

num samples train: 12544
num samples validation: 1394
saved cleaned train dataset to: data\train_cleaned.csv
saved cleaned validation dataset to: data\validation_cleaned.csv
block grouping enabled: True
current block length: 192


In [5]:
print(df_train.shape)
df_train.head()

(12544, 2)


,akkadian,english
0,<gap> ina pa-ni-ia <gap> 1 {mul}-ṣal-bat-a-nu ...,<gap> in my presence <gap> — Mars stands in Le...
1,a-na LUGAL KUR.KUR be-li₂-ia ARAD-ka {m}- <gap...,"To the king of the lands, my lord: your servan..."
2,<gap> {lu₂}-A.BA E₂—DINGIR E₂—DINGIR <gap> a-n...,"Witness NN, temple scribe. <gap> the <gap> The..."
3,{iti}-ša-ki-na-te UD.1.KAM₂ lim-mu {m}-{d}-MAŠ...,"Month of Šakināte, first day, eponymy of Nerga..."
4,IGI {m}-{iti}-AB-a.a {m}-DINGIR—ka-bar 01 UZU ...,"Witness Kanunayu. Ilu-kabar. 1 meat, 1 sheep —..."


In [6]:
df_test_cleaned = load_clean_parallel_dataset(
    TEST_SOURCE_PATH,
    TEST_TARGET_PATH,
    save_path=CLEAN_TEST_OUTPUT,
    use_blocks=USE_BLOCKS
)

df_test = df_test_cleaned.rename(columns={
    'source_text': 'akkadian',
    'target_text': 'english'
})

if MAX_TEST_SAMPLES is not None and len(df_test) > MAX_TEST_SAMPLES:
    df_test = df_test.sample(n=MAX_TEST_SAMPLES, random_state=RANDOM_STATE).reset_index(drop=True)

print('num samples test cleaned:', len(df_test_cleaned))
print('saved cleaned test dataset to:', CLEAN_TEST_OUTPUT)

num samples test cleaned: 772
saved cleaned test dataset to: data\test_cleaned.csv


In [7]:
print(df_test.shape)
df_test.head()

(772, 2)


,akkadian,english
0,<gap> {kur}-NIM.MA {ki} <gap> ig at-ta-ṣa ina ...,<gap> Elam I brought <gap> and gave <gap> wher...
1,šu₂-nu a-du <gap> a-du GUD.NITA₂-MEŠ-šu₂-nu <g...,they along with <gap> and along with their oxe...
2,ki-i pi-i ŠA₃-bi-šu₂ LUGAL be-li₂ li-pu-uš {lu...,"My messenger, who was delayed and did not come..."
3,<gap> ta-ha la u ha-ba-la i-na me-ti-iq ger-ri...,not to harm <gap> In the course of my campaign...
4,o E₂ 01 ANŠE n A.ŠA₃ ina {uru}-{lu₂}-BAHAR₂-ME...,an estate of 1 hectare and 1 decare of land in...


In [8]:
# Check for any missing values
print(f"\n🔍 Missing Values Check:")
print(f"Train akkadian missing: {df_train['akkadian'].isnull().sum()}")
print(f"Train english missing: {df_train['english'].isnull().sum()}")
print(f"Validation akkadian missing: {df_val['akkadian'].isnull().sum()}")
print(f"Validation english missing: {df_val['english'].isnull().sum()}")
print(f"Test akkadian missing: {df_test['akkadian'].isnull().sum()}")
print(f"Test english missing: {df_test['english'].isnull().sum()}")


🔍 Missing Values Check:
Train akkadian missing: 0
Train english missing: 0
Validation akkadian missing: 0
Validation english missing: 0
Test akkadian missing: 0
Test english missing: 0


In [9]:
print(f"📊 Dataset Sizes:")
print(f"Training data shape: {df_train.shape}")
print(f"Validation data shape: {df_val.shape}")
print(f"Test data shape: {df_test.shape}")
print(f"Total examples: {len(df_train) + len(df_val) + len(df_test)}")

📊 Dataset Sizes:
Training data shape: (12544, 2)
Validation data shape: (1394, 2)
Test data shape: (772, 2)
Total examples: 14710


In [10]:
# Check for GPU availability and set device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🖥️ Using device: {device}")

# Display GPU information if available
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

🖥️ Using device: cuda
GPU Name: NVIDIA GeForce RTX 4060 Laptop GPU
GPU Memory: 8.6 GB


In [11]:
# Convert pandas DataFrames to HuggingFace Dataset objects
# This format is required for the HuggingFace Trainer API
train_dataset = Dataset.from_pandas(df_train, split='train')
val_dataset = Dataset.from_pandas(df_val, split='validation')
test_dataset = Dataset.from_pandas(df_test, split='test')

print(f"✅ Created HuggingFace Datasets:")
print(f"Train dataset: {train_dataset}")
print(f"Validation dataset: {val_dataset}")
print(f"Test dataset: {test_dataset}")

✅ Created HuggingFace Datasets:
Train dataset: Dataset({
    features: ['akkadian', 'english'],
    num_rows: 12544
})
Validation dataset: Dataset({
    features: ['akkadian', 'english'],
    num_rows: 1394
})
Test dataset: Dataset({
    features: ['akkadian', 'english'],
    num_rows: 772
})


In [12]:
# Load pre-trained mBART model and tokenizer with many-to-many configuration
from transformers import MBart50TokenizerFast

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model_name = 'facebook/mbart-large-50-many-to-many-mmt'

model = MBartForConditionalGeneration.from_pretrained(model_name)
tokenizer = MBart50TokenizerFast.from_pretrained(model_name)

# Set source and target languages: use Arabic (ar_AR) as proxy for Akkadian
tokenizer.src_lang = "ar_AR"
tokenizer.tgt_lang = "en_XX"

# Add custom gap tokens as special tokens so they are never split into subwords
tokenizer.add_special_tokens({'additional_special_tokens': ["<gap>", "<big_gap>"]})

# Resize model embeddings immediately after extending the tokenizer
model.resize_token_embeddings(len(tokenizer))

# Reduce memory use during training
model.gradient_checkpointing_enable()
model.config.use_cache = False

# mBART generation needs explicit decoder start / forced BOS language tokens
if hasattr(tokenizer, "lang_code_to_id") and "en_XX" in tokenizer.lang_code_to_id:
    en_lang_id = tokenizer.lang_code_to_id["en_XX"]
    model.config.decoder_start_token_id = en_lang_id
    model.config.forced_bos_token_id = en_lang_id
    model.generation_config.decoder_start_token_id = en_lang_id
    model.generation_config.forced_bos_token_id = en_lang_id

print(f"Model parameters: {model.num_parameters():,}")
print(f"Tokenizer vocabulary size: {len(tokenizer)}")
print(f"Tokenizer type: {type(tokenizer).__name__}")
print(f"Source language: {tokenizer.src_lang}, Target language: {tokenizer.tgt_lang}")
print(f"Decoder start token id: {model.config.decoder_start_token_id}")
print(f"Forced BOS token id: {model.config.forced_bos_token_id}")
print("✅ Custom special tokens added: <gap>, <big_gap>")

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Model parameters: 610,881,536
Tokenizer vocabulary size: 250056
Tokenizer type: MBart50TokenizerFast
Source language: ar_AR, Target language: en_XX
Decoder start token id: 250004
Forced BOS token id: 250004
✅ Custom special tokens added: <gap>, <big_gap>


In [13]:
def preprocess_function(examples):
    # Keep tokenization lightweight; dynamic padding is handled by the collator.
    # Use a shorter max length in fast-test mode to keep runs short.
    inputs = tokenizer(
        examples["akkadian"],
        truncation=True,
        max_length=MAX_SOURCE_LENGTH,
        return_tensors=None
    )

    labels = tokenizer(
        examples["english"],
        truncation=True,
        max_length=MAX_TARGET_LENGTH,
        return_tensors=None
    )

    inputs["labels"] = labels["input_ids"]
    return inputs

# Tokenize training, validation, and test datasets
print("🔄 Tokenizing training dataset...")
tokenized_train_dataset = train_dataset.map(preprocess_function, batched=True, desc="Processing train")

print("🔄 Tokenizing validation dataset...")
tokenized_val_dataset = val_dataset.map(preprocess_function, batched=True, desc="Processing validation")

print("🔄 Tokenizing test dataset...")
tokenized_test_dataset = test_dataset.map(preprocess_function, batched=True, desc="Processing test")

print("✅ Tokenization complete")

🔄 Tokenizing training dataset...


Processing train:   0%|          | 0/12544 [00:00<?, ? examples/s]

🔄 Tokenizing validation dataset...


Processing validation:   0%|          | 0/1394 [00:00<?, ? examples/s]

🔄 Tokenizing test dataset...


Processing test:   0%|          | 0/772 [00:00<?, ? examples/s]

✅ Tokenization complete


In [14]:
# Set up generation config with optimized parameters for Akkadian-to-English translation

generation_config = GenerationConfig(
    max_new_tokens=BLOCK_LENGTH,
    num_beams=2,
    repetition_penalty=2.0,
    no_repeat_ngram_size=3,
    early_stopping=True,
)

training_args = Seq2SeqTrainingArguments(
    output_dir=TRAINING_OUTPUT_DIR,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.03,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    max_steps=MAX_TRAIN_STEPS,
    save_total_limit=1,
    predict_with_generate=False,
    generation_config=generation_config,
    fp16=torch.cuda.is_available(),
    eval_strategy="steps",
    eval_steps=1000,
    save_strategy="steps",
    save_steps=1000,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    optim="adafactor",
    report_to="none",
)

In [15]:
from transformers import EarlyStoppingCallback

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None,
)

# Add an early stopping callback: stop training if the evaluation metric
# (here `eval_loss`) does not improve for `early_stopping_patience` evals.
early_stopping_callback = EarlyStoppingCallback(early_stopping_patience=3)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_val_dataset,
    data_collator=data_collator,
    callbacks=[early_stopping_callback],
)

c:\Users\woutv\AppData\Local\Programs\Python\Python313\Lib\site-packages\accelerate\accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
c:\Users\woutv\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\nn\modules\module.py:1329: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  return t.to(


In [16]:
print("Ready to train Akkadian to English translation model.")

# Free up any cached GPU memory before training.
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Train the model
print("\n🚀 Starting training...")
trainer.train()

print("\n✅ Training complete!")

Ready to train Akkadian to English translation model.

🚀 Starting training...


Step,Training Loss,Validation Loss
1000,2.038100,1.894639
2000,1.535400,1.527663
3000,1.392200,1.381599
4000,1.163900,1.307465
5000,0.969300,1.265043
6000,0.986300,1.237603
7000,0.868900,1.230572
8000,0.720900,1.250429
9000,0.702800,1.218603
10000,0.611200,1.270320


c:\Users\woutv\AppData\Local\Programs\Python\Python313\Lib\site-packages\transformers\modeling_utils.py:2758: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 200, 'early_stopping': True, 'num_beams': 5, 'forced_bos_token_id': 250004}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].



✅ Training complete!


In [17]:
print(f"Model saved to: {training_args.output_dir}")

Model saved to: ./mbart-akkadian-to-en-test-len-192-full-2


In [18]:
# Quick generation-based sanity check on a small validation sample (after training)
# This cell loads your SAVED fine-tuned checkpoint if available.
from pathlib import Path
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast

checkpoint_model_dir = Path(training_args.output_dir) / "final_model"
checkpoint_tokenizer_dir = Path(training_args.output_dir) / "final_tokenizer"

if checkpoint_model_dir.exists() and checkpoint_tokenizer_dir.exists():
    print(f"Loading fine-tuned checkpoint from: {checkpoint_model_dir}")
    model = MBartForConditionalGeneration.from_pretrained(str(checkpoint_model_dir)).to(device)
    tokenizer = MBart50TokenizerFast.from_pretrained(str(checkpoint_tokenizer_dir))
else:
    print("WARNING: Fine-tuned checkpoint not found, using current in-memory model.")

# Ensure source/target language settings for mBART generation.
tokenizer.src_lang = "ar_AR"
tokenizer.tgt_lang = "en_XX"

def _ensure_generation_language_tokens(model, tokenizer, target_lang="en_XX"):
    if hasattr(tokenizer, "lang_code_to_id") and target_lang in tokenizer.lang_code_to_id:
        lang_id = tokenizer.lang_code_to_id[target_lang]
        model.config.decoder_start_token_id = lang_id
        model.config.forced_bos_token_id = lang_id
        model.generation_config.decoder_start_token_id = lang_id
        model.generation_config.forced_bos_token_id = lang_id

def translate_akkadian(akkadian_text):
    """Translate Akkadian text to English using the fine-tuned mBART model."""
    _ensure_generation_language_tokens(model, tokenizer, target_lang="en_XX")

    model.to(device)
    model.eval()

    encoded = tokenizer(akkadian_text, return_tensors="pt", max_length=BLOCK_LENGTH, truncation=True)
    encoded = encoded.to(device)

    with torch.no_grad():
        generated_tokens = model.generate(
            **encoded,
            max_new_tokens=192,
            num_beams=2,
            repetition_penalty=2.0,
            no_repeat_ngram_size=3,
            decoder_start_token_id=model.config.decoder_start_token_id,
            forced_bos_token_id=model.config.forced_bos_token_id,
        )
    translation = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]
    return translation

# Use a small fixed validation sample for quick comparison across runs
quick_val_sample = df_val.sample(n=min(8, len(df_val)), random_state=RANDOM_STATE).reset_index(drop=True)

print("🧪 Quick validation generation check:\n")
for i in range(len(quick_val_sample)):
    akkadian = quick_val_sample['akkadian'].iloc[i]
    expected_english = quick_val_sample['english'].iloc[i]
    predicted_english = translate_akkadian(akkadian)

    print(f"Example {i + 1}:")
    print(f"  Akkadian: {akkadian}")
    print(f"  Expected: {expected_english}")
    print(f"  Predicted: {predicted_english}\n")

🧪 Quick validation generation check:

Example 1:
  Akkadian: <gap> TA pa-an du-a-ki us-se-zib mu-šu an-ni-u₂ meš-hu im-šuh-ma <gap> u₂ <gap> is a-na UGU a-mat-i-ni
  Expected: <gap> cohort <gap> This night it produced a mišhu. <gap> upon our word
  Predicted: he has abandoned in the face of battle. This night was like a flashlight, and to our matter Witness NN. I am well; you can be glad. But what is it that the king, my lord, orders? who And as for the gods , his lordly majesty, whom the great gods have chosen by divine will, they will listen to his prayers. The gods Aššur, Sîn, Šamaš, Adad, and Marduk are well; may they hear your prayers! from the presence of the great judges, and may they grant you long days, everlasting years, happiness and physical well-being. With regard to this matter which concerns us, with regard to our before him ;

Example 2:
  Akkadian: du₆-li-ni ne₂-ep-pa-aš₂ ina UGU {lu₂}-DUMU—ŠAM₂-MEŠ {lu₂}-DUMU—ŠAM₂-MEŠ ša {uru}-E₂.GAL-MEŠ ša LUGAL EN iš-pur-ni-ni <gap>

In [19]:
# Save the fine-tuned model and tokenizer
print("💾 Saving fine-tuned model...")
model.save_pretrained(f"{training_args.output_dir}/final_model")
tokenizer.save_pretrained(f"{training_args.output_dir}/final_tokenizer")

print(f"✅ Model saved to: {training_args.output_dir}/final_model")
print(f"✅ Tokenizer saved to: {training_args.output_dir}/final_tokenizer")

# Optional: Load the model later with:
# from transformers import MBartForConditionalGeneration, MBart50Tokenizer
# model = MBartForConditionalGeneration.from_pretrained("{training_args.output_dir}/final_model")
# tokenizer = MBart50Tokenizer.from_pretrained("{training_args.output_dir}/final_tokenizer")

💾 Saving fine-tuned model...


c:\Users\woutv\AppData\Local\Programs\Python\Python313\Lib\site-packages\transformers\modeling_utils.py:2758: UserWarning: Moving the following attributes in the config to the generation config: {'forced_bos_token_id': 250004}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


✅ Model saved to: ./mbart-akkadian-to-en-test-len-192-full-2/final_model
✅ Tokenizer saved to: ./mbart-akkadian-to-en-test-len-192-full-2/final_tokenizer


In [22]:
print("=" * 80)
print("📊 PROJECT STATISTICS")
print("=" * 80)

print("\n🗂️  DATASET SIZES:")
print(f"   Training samples: {len(tokenized_train_dataset):,}")
print(f"   Validation samples: {len(tokenized_val_dataset):,}")
print(f"   Test samples: {len(tokenized_test_dataset):,}")
print(f"   Total samples: {len(tokenized_train_dataset) + len(tokenized_val_dataset) + len(tokenized_test_dataset):,}")

print("\n🔧 MODEL CONFIGURATION:")
print(f"   Model: {model_name}")
print(f"   Max source length: {MAX_SOURCE_LENGTH}")
print(f"   Max target length: {MAX_TARGET_LENGTH}")
print(f"   Device: {device}")
print(f"   Tokenizer type: {type(tokenizer).__name__}")

print("\n⏋ TRAINING CONFIGURATION:")
print(f"   Number of epochs: {NUM_TRAIN_EPOCHS}")
print(f"   Max training steps: {MAX_TRAIN_STEPS}")
print(f"   Early stopping enabled: {early_stopping_callback is not None}")
print(f"   Train/val split ratio: {TRAIN_SPLIT_RATIO}")

print("\n📋 TRAINING STATISTICS (from trainer):")
if trainer.state.best_metric is not None:
    print(f"   Best metric: {trainer.state.best_metric:.4f}")
print(f"   Global steps: {trainer.state.global_step}")

print("\n✨ OUTPUT PATHS:")
print(f"   Checkpoint model: {checkpoint_model_dir}")
print(f"   Checkpoint tokenizer: {checkpoint_tokenizer_dir}")
print(f"   Training output: {TRAINING_OUTPUT_DIR}")
print(f"   Clean train output: {CLEAN_TRAIN_OUTPUT}")
print(f"   Clean val output: {CLEAN_VAL_OUTPUT}")
print(f"   Clean test output: {CLEAN_TEST_OUTPUT}")

print("\n" + "=" * 80)

📊 PROJECT STATISTICS

🗂️  DATASET SIZES:
   Training samples: 12,544
   Validation samples: 1,394
   Test samples: 772
   Total samples: 14,710

🔧 MODEL CONFIGURATION:
   Model: facebook/mbart-large-50-many-to-many-mmt
   Max source length: 192
   Max target length: 192
   Device: cuda
   Tokenizer type: MBart50TokenizerFast

⏋ TRAINING CONFIGURATION:
   Number of epochs: 15
   Max training steps: -1
   Early stopping enabled: True
   Train/val split ratio: 0.9

📋 TRAINING STATISTICS (from trainer):
   Best metric: 1.2186
   Global steps: 12000

✨ OUTPUT PATHS:
   Checkpoint model: mbart-akkadian-to-en-test-len-192-full-2\final_model
   Checkpoint tokenizer: mbart-akkadian-to-en-test-len-192-full-2\final_tokenizer
   Training output: ./mbart-akkadian-to-en-test-len-192-full-2
   Clean train output: data\train_cleaned.csv
   Clean val output: data\validation_cleaned.csv
   Clean test output: data\test_cleaned.csv



In [23]:
print("=" * 80)
print("🎯 MODEL PERFORMANCE EVALUATION")
print("=" * 80)

# Evaluate on test set
print("\n🧪 Evaluating on test dataset...")
test_results = trainer.evaluate(eval_dataset=tokenized_test_dataset, metric_key_prefix="test")

print("\n📊 TEST SET METRICS:")
for key, value in test_results.items():
    if isinstance(value, float):
        print(f"   {key}: {value:.4f}")
    else:
        print(f"   {key}: {value}")

# Generate some translations on test set for qualitative evaluation
print("\n🔤 Sample translations on test set:")
sample_size = min(5, len(df_test))
test_sample = df_test.sample(n=sample_size, random_state=RANDOM_STATE).reset_index(drop=True)

for i in range(sample_size):
    akkadian = test_sample['akkadian'].iloc[i]
    expected_english = test_sample['english'].iloc[i]
    predicted_english = translate_akkadian(akkadian)
    
    print(f"\nTest Example {i + 1}:")
    print(f"  Akkadian:  {akkadian}")
    print(f"  Expected:  {expected_english}")
    print(f"  Predicted: {predicted_english}")

print("\n" + "=" * 80)

🎯 MODEL PERFORMANCE EVALUATION

🧪 Evaluating on test dataset...


early stopping required metric_for_best_model, but did not find eval_loss so early stopping is disabled



📊 TEST SET METRICS:
   test_loss: 1.2237
   test_runtime: 24.0534
   test_samples_per_second: 32.0950
   test_steps_per_second: 16.0480
   epoch: 7.6531

🔤 Sample translations on test set:

Test Example 1:
  Akkadian:  at-tu-nu ne₂-e-hu dul-la-šu₂-nu ep-pu-šu₂ TA ŠA₃ 06 {uru}-HAL.ṢU-MEŠ u₂-se-ṣi-šu₂-nu muk a-lik al-ka ia-a-mut-tu ana UGU A.ŠA₃ li-ir-ṣip lu-ši-ib 15 {lu₂}-ma-aq-tu-te ša {lu₂}-EN.NAM ša {uru}-de-ri u₂-še-bi-la-an-ni <gap> bu ma-a in-tal-ku
  Expected:  They are peaceful and do their work. I have brought them out from six forts, saying: "Go! Each one of you should build in the field and stay there!" Fifteen deserters whom the governor of Der sent to me. <gap> took counsel
  Predicted: They are working in quiet. I have brought them out of the 6 fortified cities and told them: "Go and come! Let anyone else go and build a house and settle down in the field!" 15 deserters whom the governor of Der sent to me, "They have gone" x persons Witness Šamaš-Adad-šumukin-šumi. from am